# Phase 4: Regression Analysis & Machine Learning Prediction
**Course:** ADY201m - Applied Data Analysis  
**Project:** UAE Used-Car Market Analysis & Intelligent Valuation Engine  
**Objective:** Implement baseline, OLS with HC3 robust standard errors, Random Forest, and Gradient Boosting (GBDT); evaluate across log and AED scales with Duan smearing; assign deal arbitrage flags; and produce Figures 4.1 to 4.5.

In [ ]:
import sys
from pathlib import Path
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Set project root
PROJECT_ROOT = Path("..").resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.config import (
    MODEL_FEATURES_PATH, FIGURES_DIR, RANDOM_SEED, 
    DEAL_THRESHOLD_LOG, setup_chart_style
)
from src.run_phase4 import (
    load_and_preprocess_data, get_stratified_split,
    compute_vif, run_ols_statsmodels, evaluate_predictions
)

setup_chart_style()
print("Phase 4 Environment Initialized Successfully.")

In [ ]:
# 1. Load Data & Stratified 80/20 Train/Test Split
df, top20_makes = load_and_preprocess_data()
train_df, test_df = get_stratified_split(df)

num_cols = ["car_age", "log_mileage", "cylinders", "n_features"]
bin_cols = ["has_sunroof", "has_leather_seats", "has_navigation", "has_bluetooth", "has_rear_camera", "has_adaptive_cruise"]
cat_cols = ["make_grouped", "body_type", "transmission", "fuel_type", "region_group", "condition"]
feature_cols = num_cols + bin_cols + cat_cols

print(f"Training rows: {len(train_df):,} | Test rows: {len(test_df):,}")
print(f"Total model predictors: {len(feature_cols)}")

In [ ]:
# 2. Multicollinearity Check: Variance Inflation Factors (VIF)
vif_df = compute_vif(train_df[num_cols])
print("=== Variance Inflation Factors (Numeric Predictors) ===")
display(vif_df)

In [ ]:
# 3. Fit OLS with HC3 Robust Standard Errors
model_ols, coef_df, pred_ols_tr, pred_ols_te, _ = run_ols_statsmodels(
    train_df, test_df, feature_cols, cat_cols, num_cols
)
print(f"OLS R-squared: {model_ols.rsquared:.4f} | Adj R-squared: {model_ols.rsquared_adj:.4f} | AIC: {model_ols.aic:.1f}")
print("
=== Top OLS Predictors and Percentage Effects ===")
display(coef_df.sort_values("p_value").head(15))

In [ ]:
# 4. Execute Full Modeling Pipeline & Cross-Validation
from src.run_phase4 import main as run_modeling
phase4_res = run_modeling()
print("
Phase 4 Modeling pipeline executed and saved models/price_model.joblib!")